## **Pre-trained networks, Transfer learning**

# Using ResNet for Fashion MNIST in PyTorch

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.models as models
from torchvision import transforms
import time
from tqdm.autonotebook import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import inspect

import matplotlib.pyplot as plt
import numpy as np

/Users/asa/Desktop/ML-simulator/venv_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Downloading a pre-trained network, and changing the first and last layers

The input and output layers of the pre-trained network need to be changed, since ResNet was originally designed for ImageNet competition, which was a color (3-channel) image classification task with 1000 classes.

MNIST dataset howerver only contains 10 classes and it’s images are in the grayscale (1-channel)

In [20]:
class MnistResNet(nn.Module):
  def __init__(self, in_channels=1):
    super(MnistResNet, self).__init__()

    # Load a pretrained resnet model from torchvision.models in Pytorch
    self.model = models.resnet50(weights="ResNet50_Weights.DEFAULT")

    # Change the input layer to take Grayscale image, instead of RGB images.
    # Hence in_channels is set as 1 or 3 respectively
    # original definition of the first layer on the ResNet class
    # self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    self.model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

    # Change the output layer to output 10 classes instead of 1000 classes
    num_ftrs = self.model.fc.in_features
    self.model.fc = nn.Linear(num_ftrs, 10)

  def forward(self, x):
    return self.model(x)


my_resnet = MnistResNet()



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/asa/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [01:09<00:00, 1.48MB/s]


### Test defined network, and verify layers


In [21]:
input = torch.randn((16,1,244,244))
output = my_resnet(input)
print(output.shape)

print(my_resnet)

# Print the number of parameters in the model
total_params = sum(p.numel() for p in my_resnet.parameters())
print(f"Total number of parameters: {total_params}")


torch.Size([16, 10])
MnistResNet(
  (model): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample):

### Define device

In [49]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

### Dataloaders


In [83]:
def get_data_loaders(train_batch_size, val_batch_size):
    fashion_mnist = torchvision.datasets.FashionMNIST(download=True, train=True, root=".").train_data.float()

    data_transform = transforms.Compose([transforms.Resize((224, 224)),
                                         transforms.ToTensor(),
                                         transforms.Normalize((fashion_mnist.mean()/255,), (fashion_mnist.std()/255,))])

    train_loader = DataLoader(torchvision.datasets.FashionMNIST(download=True, root=".", transform=data_transform, train=True),
                              batch_size=train_batch_size, shuffle=True)

    val_loader = DataLoader(torchvision.datasets.FashionMNIST(download=False, root=".", transform=data_transform, train=False),
                            batch_size=val_batch_size, shuffle=False)
    return train_loader, val_loader

### Supporting functions for metric calculation

In [84]:
def calculate_metric(metric_fn, true_y, pred_y):
    if metric_fn.__name__ == 'accuracy_score':
        return metric_fn(true_y, pred_y)
    return metric_fn(true_y, pred_y, average="macro")

def print_scores(p, r, f1, a, batch_size):
    for name, scores in zip(("precision", "recall", "F1", "accuracy"), (p, r, f1, a)):
        print(f"\t{name.rjust(14, ' ')}: {sum(scores)/batch_size:.4f}")

### Pytorch Deep Learning Boilerplate

Boilerplate are the sections of code that have to be included in many places with little or no alteration

In [85]:
def train_model(model, train_loader, val_loader, epochs, loss_function, optimizer, device):
    model = model.to(device)

    start_ts = time.time()
    losses = []
    batches = len(train_loader)
    val_batches = len(val_loader)

    # loop for every epoch (training + evaluation)
    for epoch in range(epochs):
        total_loss = 0

        # progress bar (works in Jupyter notebook too!)
        progress = tqdm(enumerate(train_loader), desc="Loss: ", total=batches)

        # ----------------- TRAINING  --------------------
        # set model to training
        model.train()

        for i, data in progress:
            X, y = data[0].to(device), data[1].to(device)

            # training step for single batch
            model.zero_grad()
            outputs = model(X)
            loss = loss_function(outputs, y)
            loss.backward()
            optimizer.step()

            # getting training quality data
            current_loss = loss.item()
            total_loss += current_loss

            # updating progress bar
            progress.set_description("Loss: {:.4f}".format(total_loss / (i + 1)))

        # releasing unnecessary memory in GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ----------------- VALIDATION  -----------------
        val_losses = 0
        precision, recall, f1, accuracy = [], [], [], []

        # set model to evaluating (testing)
        model.eval()
        with torch.no_grad():
            for i, data in tqdm(enumerate(val_loader), desc="Validation: ", total=val_batches):
                X, y = data[0].to(device), data[1].to(device)

                outputs = model(X)  # this gets the prediction from the network

                val_losses += loss_function(outputs, y)

                predicted_classes = torch.max(outputs, 1)[
                    1
                ]  # get class from network's prediction

                # calculate P/R/F1/A metrics for batch
                for acc, metric in zip(
                    (precision, recall, f1, accuracy),
                    (precision_score, recall_score, f1_score, accuracy_score),
                ):
                    acc.append(calculate_metric(metric, y.cpu(), predicted_classes.cpu()))

        print(
            f"Epoch {epoch+1}/{epochs}, training loss: {total_loss/batches}, validation loss: {val_losses/val_batches}"
        )
        print_scores(precision, recall, f1, accuracy, val_batches)
        losses.append(total_loss / batches)  # for plotting learning curve

        # Save checkpoint after each epoch
        checkpoint_path = f"checkpoint_epoch_{epoch+1}.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": total_loss / batches,
            },
            checkpoint_path,
        )

    print(f"Training time: {time.time() - start_ts}s")

# model:
model = MnistResNet().to(device)

# params you need to specify:
epochs = 3
batch_size = 64

# Dataloaders
train_loader, val_loader = get_data_loaders(batch_size, batch_size)

# loss function and optimizer
loss_function = (
    nn.CrossEntropyLoss()
)  # your loss function, cross entropy works well for multi-class problems

# optimizer, I've used Adadelta, as it works well without any magic numbers
optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)  # Using Karpathy's learning rate constant

# train_model(model, train_loader, val_loader, epochs, loss_function, optimizer, device)


100%|██████████| 26421880/26421880 [00:14<00:00, 1780460.10it/s]


Extracting ./FashionMNIST/raw/train-images-idx3-ubyte.gz to ./FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 353032.94it/s]


Extracting ./FashionMNIST/raw/train-labels-idx1-ubyte.gz to ./FashionMNIST/raw



100%|██████████| 4422102/4422102 [00:01<00:00, 2642313.84it/s]


Extracting ./FashionMNIST/raw/t10k-images-idx3-ubyte.gz to ./FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 16609443.84it/s]

Extracting ./FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to ./FashionMNIST/raw




/Users/asa/Desktop/ML-simulator/venv_new/lib/python3.10/site-packages/torchvision/datasets/mnist.py:75: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")


### Save Model

In [ ]:
torch.save(model.state_dict(), "parent_model.pth")

model = MnistResNet()
model_state_dict = torch.load("parent_model.pth")
model.load_state_dict(model_state_dict)

<All keys matched successfully>

## Distillation

In [94]:
class DistilModel(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(DistilModel, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            64,
            kernel_size=(7, 7),
            stride=(2, 2),
            padding=(3, 3),
            bias=False,
        )
        self.bn = nn.BatchNorm2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
        )
        self.relu = nn.ReLU(inplace=False)
        self.maxpool = nn.MaxPool2d(
            kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False
        )
        self.avgpool = nn.AdaptiveAvgPool2d((8, 4))

        self.fc1 = nn.Linear(64 * 8 * 4, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        fc1_output = self.fc1(x)
        predictions = self.softmax(fc1_output)
        return predictions, fc1_output


In [92]:
distil_model = DistilModel()

input = torch.randn((1, 1, 244, 244))
pred, conv1_output = distil_model(input)

print(distil_model)

total_params = sum(p.numel() for p in distil_model.parameters())
print(f"Number of parameters in distil_model: {total_params}")


DistilModel(
  (conv): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (avgpool): AdaptiveAvgPool2d(output_size=(8, 4))
  (fc1): Linear(in_features=2048, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)
Number of parameters in distil_model: 2631370


In [77]:
device

device(type='cpu')

In [59]:
torch.save(distil_model.state_dict(), "distil_model.pth")

In [95]:

# params you need to specify:
epochs = 3
batch_size = 64

# Dataloaders
train_loader, val_loader = get_data_loaders(batch_size, batch_size)

# loss function and optimizer
loss_function = (
    nn.CrossEntropyLoss()
)  # your loss function, cross entropy works well for multi-class problems

# optimizer, I've used Adadelta, as it works well without any magic numbers
optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)  # Using Karpathy's learning rate constant

# train_model(
#     distil_model, train_loader, val_loader, epochs, loss_function, optimizer, device
# )


In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [87]:
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

class ParentModel(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(ParentModel, self).__init__()

        # Load a pretrained resnet model from torchvision.models in Pytorch
        self.model = models.resnet50(weights="ResNet50_Weights.DEFAULT")

        # Change the input layer to take Grayscale image, instead of RGB images.
        # Hence in_channels is set as 1 or 3 respectively
        # original definition of the first layer on the ResNet class
        # self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.conv1 = nn.Conv2d(
            in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # Change the output layer to output 10 classes instead of 1000 classes
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

        # Register hook to capture the output of the fully connected layer
        self.model.fc.register_forward_hook(get_activation('fc'))

    def forward(self, x):
        predictions = self.model(x)
        fc_output = activation['fc']
        return predictions, fc_output



In [96]:
def distil_train(
    teacher,
    student,
    train_loader,
    val_loader,  # Added val_loader parameter
    epochs,
    learning_rate,
    feature_map_weight,
    ce_loss_weight,
    device,
):
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()  # Teacher set to evaluation mode
    student.train()  # Student to train mode

    batches = len(train_loader)

    for epoch in range(epochs):
        running_loss = 0.0
        all_labels = []
        all_predictions = []
        progress = tqdm(
            enumerate(train_loader), desc=f"Epoch {epoch+1}/{epochs}: ", total=batches
        )
        for i, data in progress:
            inputs, labels = data[0].to(device), data[1].to(device)

            optimizer.zero_grad()

            # Get teacher's feature map
            with torch.no_grad():
                _, teacher_feature_map = teacher(inputs)

            # Forward pass with the student model
            student_logits, student_feature_map = student(inputs)

            # Calculate the loss
            hidden_rep_loss = mse_loss(student_feature_map, teacher_feature_map)
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Collect labels and predictions for accuracy calculation
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(student_logits.argmax(dim=1).cpu().numpy())

        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = accuracy_score(all_labels, all_predictions)
        print(
            f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss}, Accuracy: {epoch_accuracy}"
        )

        # Validation accuracy calculation
        student.eval()
        val_labels = []
        val_predictions = []
        val_batches = len(val_loader)
        with torch.no_grad():
            for i, val_data in tqdm(
                enumerate(val_loader), desc="Validation: ", total=val_batches
            ):
                val_inputs, val_labels_batch = (
                    val_data[0].to(device),
                    val_data[1].to(device),
                )
                val_logits, _ = student(val_inputs)
                val_labels.extend(val_labels_batch.cpu().numpy())
                val_predictions.extend(val_logits.argmax(dim=1).cpu().numpy())
        val_accuracy = accuracy_score(val_labels, val_predictions)
        print(f"Validation Accuracy: {val_accuracy}")
        student.train()


In [98]:
# Example usage
torch.manual_seed(42)
distil_model = DistilModel(num_classes=10).to(device)
parent_model = ParentModel(num_classes=10).to(device)
parent_model.load_state_dict(
    torch.load("parent_model.pth", map_location=device)
)

distil_train(
    teacher=parent_model,
    student=distil_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=3,
    learning_rate=0.003,
    feature_map_weight=0.25,
    ce_loss_weight=0.75,
    device=device,
)


Epoch 1/3:   0%|          | 2/938 [00:17<2:15:58,  8.72s/it]


KeyboardInterrupt: 